# K-bin CRE sensitivity, revised subject-level OOF version

This notebook replaces the old `k_chain_rule.ipynb` input with the revised prediction tables:

- Full cohort: `autorater_predictions_all_subject_oof_with_scan_age.csv`
- Age 65--70 subset: `autorater_predictions_6570_subject_oof_scan_age.csv`

It runs a conjugate K-bin Bayesian chain-rule sensitivity analysis:

\[
g = \sum_{k=1}^{K} P(B=k) P(H=1 \mid B=k)
\]

where \(B\) is a bin of the continuous autorater probability.  
This is a sensitivity analysis, not the main repeated-labeling coverage experiment.

In [ ]:
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


In [ ]:
%pip install -q pandas numpy matplotlib scipy tqdm

In [ ]:
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# ============================================================
# 0) Paths and settings
# ============================================================

FULL_CSV = REPO_ROOT / "results/autorater/autorater_predictions_all_subject_oof_with_scan_age.csv"
AGE6570_CSV = REPO_ROOT / "results/autorater/autorater_predictions_6570_subject_oof_scan_age.csv"

OUT_DIR = REPO_ROOT / "results/kbin"
FIG_DIR = OUT_DIR / "figures"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "full": FULL_CSV,
    "age_65_70": AGE6570_CSV,
}

DISPLAY_NAME = {
    "full": "Full cohort",
    "age_65_70": "Age 65--70 subset",
}

K_LIST = [2, 4, 5]
BINNING_STRATEGY: Literal["quantile", "uniform"] = "quantile"

PRIORS = [
    {"name": "uniform", "alpha_dirichlet": 1.0, "a_beta": 1.0, "b_beta": 1.0},
    {"name": "jeffreys", "alpha_dirichlet": 0.5, "a_beta": 0.5, "b_beta": 0.5},
]

N_DRAWS = 10000
SEED = 2025

print("OUT_DIR:", OUT_DIR)

In [ ]:
# ============================================================
# 1) Load revised prediction tables
# ============================================================

def load_prediction_csv(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"Could not find: {csv_path}")

    df = pd.read_csv(csv_path).copy()
    df.columns = [c.strip() for c in df.columns]

    if "H" not in df.columns:
        if "label" not in df.columns:
            raise ValueError("Need either H or label column.")
        df["H"] = (df["label"].astype(str).str.upper().str.strip() == "AD").astype(int)
    else:
        df["H"] = pd.to_numeric(df["H"], errors="coerce").astype(int)

    if "autorater_prediction" not in df.columns:
        raise ValueError("Missing autorater_prediction column.")

    df["A_prob"] = pd.to_numeric(df["autorater_prediction"], errors="coerce")
    df = df.dropna(subset=["A_prob", "H"]).copy()
    df = df[df["A_prob"].between(0, 1, inclusive="both")].copy()

    if "subject_id" in df.columns:
        df["subject_id"] = df["subject_id"].astype(str).str.strip()

    if "Age" in df.columns:
        df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

    return df.reset_index(drop=True)


def qc_dataset(df: pd.DataFrame, name: str) -> None:
    print("=" * 90)
    print("QC:", name)
    print("=" * 90)
    print("Rows:", len(df))
    if "subject_id" in df.columns:
        print("Unique subjects:", df["subject_id"].nunique())
    print("True prevalence:", float(df["H"].mean()))
    print("A_prob range:", float(df["A_prob"].min()), "to", float(df["A_prob"].max()))
    if "Age" in df.columns:
        print("Age range:", float(df["Age"].min()), "to", float(df["Age"].max()))
    if "oof_fold" in df.columns and "subject_id" in df.columns:
        per_subject_fold_n = df.groupby("subject_id")["oof_fold"].nunique()
        print("Subjects spanning multiple OOF folds:", int((per_subject_fold_n > 1).sum()))
        print("Max folds per subject:", int(per_subject_fold_n.max()))


datasets = {}
for dataset_name, csv_path in DATASETS.items():
    datasets[dataset_name] = load_prediction_csv(csv_path)
    qc_dataset(datasets[dataset_name], DISPLAY_NAME.get(dataset_name, dataset_name))

In [ ]:
# ============================================================
# 2) K-bin CRE utilities
# ============================================================

def assign_bins_from_full_pool(
    scores: pd.Series,
    K: int,
    strategy: Literal["quantile", "uniform"] = "quantile",
):
    """
    Assign bins using the full score pool.

    Returns:
      bin codes for each score
      bin edges
      number of effective bins
    """
    scores = pd.to_numeric(scores, errors="coerce")

    if strategy == "quantile":
        codes, edges = pd.qcut(
            scores,
            q=min(K, scores.nunique()),
            labels=False,
            duplicates="drop",
            retbins=True,
        )
    elif strategy == "uniform":
        codes, edges = pd.cut(
            scores,
            bins=K,
            labels=False,
            include_lowest=True,
            retbins=True,
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    codes = pd.Series(codes, index=scores.index)
    K_eff = int(codes.nunique(dropna=True))

    return codes, np.asarray(edges, dtype=float), K_eff


def assign_bins_from_edges(scores: pd.Series, edges: np.ndarray):
    """
    Reuse bin edges estimated from the full pool.
    The first/last edges are widened slightly to include boundary values.
    """
    edges = np.asarray(edges, dtype=float).copy()
    edges[0] = -np.inf
    edges[-1] = np.inf

    codes = pd.cut(
        pd.to_numeric(scores, errors="coerce"),
        bins=edges,
        labels=False,
        include_lowest=True,
    )
    return pd.Series(codes, index=scores.index)


def posterior_draws_kbin(
    df: pd.DataFrame,
    K: int,
    strategy: Literal["quantile", "uniform"],
    alpha_dirichlet: float,
    a_beta: float,
    b_beta: float,
    S: int,
    rng: np.random.Generator,
):
    """
    Conjugate Bayesian K-bin chain-rule estimator:

      pi = P(B=k)                 Dirichlet posterior
      theta_k = P(H=1 | B=k)      independent Beta posteriors
      g = sum_k pi_k theta_k
    """
    full = df.dropna(subset=["A_prob"]).copy()
    full["bin"], edges, K_eff = assign_bins_from_full_pool(full["A_prob"], K, strategy=strategy)

    if K_eff < 2:
        raise ValueError(f"Effective bins is {K_eff}. Try a smaller K or uniform binning.")

    counts_B = (
        full["bin"]
        .value_counts()
        .sort_index()
        .reindex(range(K_eff), fill_value=0)
        .astype(int)
    )
    alpha_vec = np.full(K_eff, alpha_dirichlet, dtype=float) + counts_B.to_numpy(dtype=float)

    lab = df.dropna(subset=["A_prob", "H"]).copy()
    lab["bin"] = assign_bins_from_edges(lab["A_prob"], edges)
    lab = lab.dropna(subset=["bin"]).copy()
    lab["bin"] = lab["bin"].astype(int)

    n_pos = (
        lab.groupby("bin")["H"]
        .sum()
        .reindex(range(K_eff), fill_value=0)
        .astype(int)
    )
    n_total = (
        lab.groupby("bin")["H"]
        .count()
        .reindex(range(K_eff), fill_value=0)
        .astype(int)
    )

    a_post = a_beta + n_pos.to_numpy(dtype=float)
    b_post = b_beta + (n_total - n_pos).to_numpy(dtype=float)

    pi_draws = rng.dirichlet(alpha_vec, size=S)
    theta_draws = rng.beta(a_post, b_post, size=(S, K_eff))
    g_draws = np.sum(pi_draws * theta_draws, axis=1)

    lo, hi = np.quantile(g_draws, [0.025, 0.975])

    return {
        "K_requested": K,
        "K_effective": K_eff,
        "strategy": strategy,
        "mean": float(g_draws.mean()),
        "sd": float(g_draws.std(ddof=1)),
        "ci_low": float(lo),
        "ci_high": float(hi),
        "width": float(hi - lo),
        "bin_edges": ";".join([f"{x:.8g}" for x in edges]),
        "min_bin_count_full": int(counts_B.min()),
        "min_bin_count_labeled": int(n_total.min()),
        "max_bin_count_labeled": int(n_total.max()),
    }

In [ ]:
# ============================================================
# 3) Run K-bin sensitivity
# ============================================================

def run_kbin_sensitivity_for_dataset(
    df: pd.DataFrame,
    dataset_name: str,
    csv_path: Path,
    out_dir: Path,
):
    rows = []

    for prior in PRIORS:
        for K in K_LIST:
            local_seed = SEED + 1000 * K + (0 if prior["name"] == "uniform" else 100000)
            rng = np.random.default_rng(local_seed)

            res = posterior_draws_kbin(
                df=df,
                K=K,
                strategy=BINNING_STRATEGY,
                alpha_dirichlet=prior["alpha_dirichlet"],
                a_beta=prior["a_beta"],
                b_beta=prior["b_beta"],
                S=N_DRAWS,
                rng=rng,
            )

            res.update({
                "dataset": dataset_name,
                "input_csv_path": str(csv_path),
                "n_rows": len(df),
                "n_subjects": df["subject_id"].nunique() if "subject_id" in df.columns else np.nan,
                "g_true_empirical": float(df["H"].mean()),
                "prior": prior["name"],
                "alpha_dirichlet": prior["alpha_dirichlet"],
                "a_beta": prior["a_beta"],
                "b_beta": prior["b_beta"],
                "n_draws": N_DRAWS,
            })
            rows.append(res)

            print(
                f"{dataset_name} | {prior['name']} | K={K} "
                f"(eff={res['K_effective']}): mean={res['mean']:.4f}, "
                f"CI=[{res['ci_low']:.4f}, {res['ci_high']:.4f}], "
                f"width={res['width']:.4f}"
            )

    out = pd.DataFrame(rows)

    out["delta_mean_vs_K2"] = np.nan
    out["width_ratio_vs_K2"] = np.nan

    for prior_name in out["prior"].unique():
        mask = out["prior"] == prior_name
        ref = out.loc[mask & (out["K_requested"] == 2)]
        if len(ref) == 1:
            ref_mean = float(ref["mean"].iloc[0])
            ref_width = float(ref["width"].iloc[0])
            out.loc[mask, "delta_mean_vs_K2"] = out.loc[mask, "mean"] - ref_mean
            out.loc[mask, "width_ratio_vs_K2"] = out.loc[mask, "width"] / ref_width

    cols = [
        "dataset", "input_csv_path", "n_rows", "n_subjects", "g_true_empirical",
        "prior", "K_requested", "K_effective", "strategy",
        "mean", "sd", "ci_low", "ci_high", "width",
        "delta_mean_vs_K2", "width_ratio_vs_K2",
        "min_bin_count_full", "min_bin_count_labeled", "max_bin_count_labeled",
        "alpha_dirichlet", "a_beta", "b_beta", "n_draws", "bin_edges",
    ]
    out = out[cols]

    out_csv = out_dir / f"kbin_sensitivity_{dataset_name}.csv"
    out.to_csv(out_csv, index=False)

    return out


all_kbin = []
for dataset_name, df in datasets.items():
    res = run_kbin_sensitivity_for_dataset(
        df=df,
        dataset_name=dataset_name,
        csv_path=DATASETS[dataset_name],
        out_dir=OUT_DIR,
    )
    all_kbin.append(res)

kbin_summary = pd.concat(all_kbin, ignore_index=True)
combined_csv = OUT_DIR / "kbin_sensitivity_summary_revised.csv"
kbin_summary.to_csv(combined_csv, index=False)

print("Saved combined K-bin summary:", combined_csv)
display(kbin_summary.round(4))

In [ ]:
# ============================================================
# 4) Paper-friendly summary
# ============================================================

paper_summary = kbin_summary[
    [
        "dataset", "prior", "K_requested", "K_effective",
        "mean", "ci_low", "ci_high", "width",
        "delta_mean_vs_K2", "width_ratio_vs_K2",
        "min_bin_count_labeled",
    ]
].copy()

paper_summary.to_csv(OUT_DIR / "kbin_paper_summary_revised.csv", index=False)

display(paper_summary.round(4))

In [ ]:
# ============================================================
# 5) Figures
# ============================================================

def plot_kbin_width(summary: pd.DataFrame, dataset_name: str):
    sub = summary[summary["dataset"] == dataset_name].copy()
    title_name = DISPLAY_NAME.get(dataset_name, dataset_name)

    plt.figure(figsize=(6, 4))

    for prior_name in ["uniform", "jeffreys"]:
        psub = sub[sub["prior"] == prior_name].sort_values("K_requested")
        plt.plot(psub["K_requested"], psub["width"], marker="o", label=prior_name)

    plt.xlabel("K requested")
    plt.ylabel("95% posterior interval width")
    plt.title(f"K-bin CRE sensitivity: {title_name}")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    out_png = FIG_DIR / f"fig_kbin_width_{dataset_name}.png"
    out_pdf = FIG_DIR / f"fig_kbin_width_{dataset_name}.pdf"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)
    print("Saved:", out_pdf)


def plot_kbin_mean(summary: pd.DataFrame, dataset_name: str):
    sub = summary[summary["dataset"] == dataset_name].copy()
    title_name = DISPLAY_NAME.get(dataset_name, dataset_name)

    plt.figure(figsize=(6, 4))

    for prior_name in ["uniform", "jeffreys"]:
        psub = sub[sub["prior"] == prior_name].sort_values("K_requested")
        plt.plot(psub["K_requested"], psub["mean"], marker="o", label=prior_name)

    g_emp = float(sub["g_true_empirical"].iloc[0])
    plt.axhline(g_emp, linestyle="--", linewidth=1, label="Empirical H prevalence")

    plt.xlabel("K requested")
    plt.ylabel("Posterior mean of g")
    plt.title(f"K-bin CRE posterior mean: {title_name}")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    out_png = FIG_DIR / f"fig_kbin_mean_{dataset_name}.png"
    out_pdf = FIG_DIR / f"fig_kbin_mean_{dataset_name}.pdf"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)
    print("Saved:", out_pdf)


for dataset_name in datasets.keys():
    plot_kbin_width(kbin_summary, dataset_name)
    plot_kbin_mean(kbin_summary, dataset_name)

## Manuscript wording

Use this conservatively:

> We include K-bin CRE as a sensitivity analysis using quantile bins of the continuous autorater score. Increasing from \(K=2\) to \(K=4\) or \(K=5\) did not materially change the posterior mean of \(g\), and interval widths changed only modestly under the available labeled sample sizes. We therefore keep the thresholded \(K=2\) deployment estimand as the main analysis and treat multi-bin variants as sensitivity checks rather than a fully exercised non-conjugate extension.